## **EDA**

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(),"..", "src"))
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import build_dataset as bd
import importlib
importlib.reload(bd)

from build_dataset import TARGET, LABEL_MAP

pd.set_option("display.max_colwidth", None)

**1. Understanding attack types present in dataset**

`label_by_file` lists the unique attack types and the number of rows present in each original data file. 
`label_distribution`

In [2]:
# DETERMINE ATTACK TYPES APPEARING IN EACH FILE WITH COUNTS
bd.label_by_file()

,file,n_rows,attack_types
0,Monday-WorkingHours.pcap_ISCX.csv,502573,BENIGN
1,Tuesday-WorkingHours.pcap_ISCX.csv,414134,"BENIGN, FTP-Patator, SSH-Patator"
2,Wednesday-workingHours.pcap_ISCX.csv,594193,"BENIGN, DoS GoldenEye, DoS Hulk, DoS Slowhttptest, DoS slowloris, Heartbleed"
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,154104,"BENIGN, Web Attack - Brute Force, Web Attack - Sql Injection, Web Attack - XSS"
4,Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,240082,"BENIGN, Infiltration"
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,171359,"BENIGN, Bot"
6,Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,204230,"BENIGN, PortScan"
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,217305,"BENIGN, DDoS"


In [3]:
# DETERMINE FREQUENCY OF EACH LABEL
bd.label_distribution()

,Label,Count,Percentage
0,BENIGN,2072254,82.96
1,DoS Hulk,172846,6.92
2,DDoS,128014,5.12
3,PortScan,90694,3.63
4,DoS GoldenEye,10282,0.41
5,FTP-Patator,5931,0.24
6,DoS slowloris,5374,0.22
7,DoS Slowhttptest,5228,0.21
8,SSH-Patator,3219,0.13
9,Bot,1948,0.08


Labels representing similar attacks are grouped together. This is because some classes are too small to evaluate on their own, and several labels represent the same type of attack. Infiltration and Heartbleed are excluded due to lack of data. 

1. DDoS
2. DoS
    DoS Hulk, DoS Golden Eye, DoS slowloris, DoS Slowhttptest
3. PortScan
4. Bot
5. BruteForce
    FTP-Patator, SSH-Patator
6. WebAttack
    Web Attack - Brute Force, Web Attack - XSS, Web Attack - Sql Injection
7. EXCLUDED
    Infiltration [36 attacks], Heartbleed [11 attacks]

In [4]:
# MAP LABELS TO GROUPS
LABEL_MAP

{'BENIGN': 'BENIGN',
 'DDoS': 'DDoS',
 'DoS Hulk': 'DoS',
 'DoS GoldenEye': 'DoS',
 'DoS slowloris': 'DoS',
 'DoS Slowhttptest': 'DoS',
 'PortScan': 'PortScan',
 'Bot': 'Bot',
 'FTP-Patator': 'BruteForce',
 'SSH-Patator': 'BruteForce',
 'Web Attack - Brute Force': 'WebAttack',
 'Web Attack - XSS': 'WebAttack',
 'Web Attack - Sql Injection': 'WebAttack',
 'Infiltration': 'EXCLUDED',
 'Heartbleed': 'EXCLUDED'}

In [5]:
df = bd.load_dataset(columns=[TARGET, "SourceFile"], label_group=True, drop_excluded=True)
print(df["LabelGroup"].value_counts(dropna=False))

LabelGroup
BENIGN        2072254
DoS            193730
DDoS           128014
PortScan        90694
BruteForce       9150
WebAttack        2143
Bot              1948
Name: count, dtype: int64
